# Data Preprocessing and Feature Engineering

## 1. Preprocessing Overview

This notebook continues from the Exploratory Data Analysis (EDA) performed on the USA Housing Listings dataset.

The preprocessing decisions are based on the findings identified during EDA. The objective is to transform the raw dataset into a clean, consistent, and model-ready dataset while preserving useful information and preventing data leakage.

The main preprocessing tasks include:

- Handling invalid records
- Handling missing values
- Investigating repeated listings
- Treating extreme numerical observations where justified
- Removing leakage-prone or unusable features
- Feature engineering
- Encoding categorical variables
- Scaling numerical variables where required
- Feature selection
- Building a reproducible preprocessing pipeline
- Validating the final processed dataset

The target variable for the regression task is `price`.

## Preprocessing Workflow

1. Load the dataset and create a working copy
2. Define the target variable and feature groups
3. Remove clearly invalid records
4. Remove leakage-prone and unusable features
5. Investigate repeated listing patterns
6. Separate training, validation, and test data
7. Handle missing values
8. Treat extreme numerical observations
9. Perform feature engineering
10. Encode categorical variables
11. Scale numerical variables where required
12. Perform feature selection
13. Build the preprocessing pipeline
14. Validate the processed data
15. Perform a final data leakage audit

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load the original dataset
raw_df = pd.read_csv("../data/housing.csv")

# Create a separate working copy for preprocessing
df = raw_df.copy()

# Check the shapes
print("Raw dataset shape:", raw_df.shape)
print("Working dataset shape:", df.shape)

Raw dataset shape: (384977, 22)
Working dataset shape: (384977, 22)


## 2. Define Target and Feature Groups

Based on the EDA findings, the target variable is `price`, representing the advertised monthly rental price.

The remaining variables are grouped according to their data type and modeling role. These groups will be used to guide the preprocessing decisions in later sections.

In [4]:
# Define the target variable
target = "price"

# Numerical features
numerical_features = [
    "sqfeet",
    "beds",
    "baths",
    "lat",
    "long"
]

# Binary features
binary_features = [
    "cats_allowed",
    "dogs_allowed",
    "smoking_allowed",
    "wheelchair_access",
    "electric_vehicle_charge",
    "comes_furnished"
]

# Categorical features
categorical_features = [
    "type",
    "region",
    "state",
    "laundry_options",
    "parking_options"
]

# Listing text and identifier/URL columns
excluded_features = [
    "id",
    "url",
    "region_url",
    "image_url",
    "description"
]

# Display the feature groups
print("Target:", target)
print("\nNumerical features:", numerical_features)
print("\nBinary features:", binary_features)
print("\nCategorical features:", categorical_features)
print("\nExcluded features:", excluded_features)

Target: price

Numerical features: ['sqfeet', 'beds', 'baths', 'lat', 'long']

Binary features: ['cats_allowed', 'dogs_allowed', 'smoking_allowed', 'wheelchair_access', 'electric_vehicle_charge', 'comes_furnished']

Categorical features: ['type', 'region', 'state', 'laundry_options', 'parking_options']

Excluded features: ['id', 'url', 'region_url', 'image_url', 'description']


## 3. Remove Clearly Invalid Records

The EDA identified several records that violate basic validity constraints for the rental-price prediction problem.

The following records will be removed:

- Listings with `price <= 0`, because the target represents a monthly rental price.
- Listings with `sqfeet <= 0`, because a property cannot have zero or negative usable square footage.
- Listings with latitude outside the valid geographic range of -90 to 90 degrees.

No negative bedroom or bathroom values were found, so no removal is required for those variables at this stage.

Extreme but potentially valid observations such as unusually high rental prices, square footage, bedrooms, or bathrooms will be investigated separately rather than automatically removed here.

In [5]:
# Count clearly invalid records identified during EDA

invalid_price = df["price"] <= 0
invalid_sqfeet = df["sqfeet"] <= 0

# Only check latitude values that are actually present.
# Missing latitude values will be handled later as missing data.
invalid_lat = (
    df["lat"].notna() &
    ~df["lat"].between(-90, 90)
)

print("Invalid price records:", invalid_price.sum())
print("Invalid square-footage records:", invalid_sqfeet.sum())
print("Invalid latitude records:", invalid_lat.sum())

# Combine all clearly invalid conditions
invalid_any = invalid_price | invalid_sqfeet | invalid_lat

print("\nTotal records flagged for removal:", invalid_any.sum())

Invalid price records: 1307
Invalid square-footage records: 48
Invalid latitude records: 4

Total records flagged for removal: 1359


In [6]:
# Remove records that violate basic validity rules
df = df.loc[~invalid_any].copy()

# Check the new dataset size
print("Dataset shape after removing invalid records:", df.shape)

# Confirm that the invalid records are gone
print("\nRemaining invalid price records:", (df["price"] <= 0).sum())
print("Remaining invalid square-footage records:", (df["sqfeet"] <= 0).sum())

remaining_invalid_lat = (
    df["lat"].notna() &
    ~df["lat"].between(-90, 90)
)

print("Remaining invalid latitude records:", remaining_invalid_lat.sum())

Dataset shape after removing invalid records: (383618, 22)

Remaining invalid price records: 0
Remaining invalid square-footage records: 0
Remaining invalid latitude records: 0


## 4. Remove Leakage and Unusable Features

Several columns were excluded based on the EDA and leakage analysis.

- `id` is an identifier rather than a property characteristic.
- `url`, `region_url`, and `image_url` are listing metadata and are not used as predictive features.
- `description` is excluded because the EDA found that many descriptions contain explicit dollar-price information, including values matching the target variable `price`. Using this column could cause target leakage.

These columns are therefore removed before model development.

In [7]:
# Columns identified as unusable or potentially leaking the target
columns_to_remove = [
    "id",
    "url",
    "region_url",
    "image_url",
    "description"
]

# Remove the columns from the working dataset
df = df.drop(columns=columns_to_remove)

print("Dataset shape after removing excluded columns:", df.shape)

print("\nRemaining columns:")
print(df.columns.tolist())

Dataset shape after removing excluded columns: (383618, 17)

Remaining columns:
['region', 'price', 'type', 'sqfeet', 'beds', 'baths', 'cats_allowed', 'dogs_allowed', 'smoking_allowed', 'wheelchair_access', 'electric_vehicle_charge', 'comes_furnished', 'laundry_options', 'parking_options', 'lat', 'long', 'state']


## 5. Handle Missing Values

The EDA identified missing values in `parking_options`, `laundry_options`, `lat`, and `long`.

The categorical missing values will be considered separately from the geographic missing values because missing facility information does not necessarily mean that the facility is absent.

Latitude and longitude will also be handled separately rather than being replaced with arbitrary mean coordinates.

First, the remaining missing values are inspected after the invalid-record and leakage-removal steps.

In [8]:
# Check the number of missing values in each remaining column
missing_counts = df.isnull().sum()

# Display only columns that contain missing values
missing_counts = missing_counts[missing_counts > 0]

print("Remaining missing values:")
print(missing_counts)

print("\nMissing-value percentages:")
print((missing_counts / len(df) * 100).round(2))

Remaining missing values:
laundry_options     78819
parking_options    140289
lat                  1909
long                 1909
dtype: int64

Missing-value percentages:
laundry_options    20.55
parking_options    36.57
lat                 0.50
long                0.50
dtype: float64


### 5.1 Handle Missing Categorical Values

The `parking_options` and `laundry_options` columns contain a substantial proportion of missing values.

Missing values are not assumed to mean that the facility is unavailable because the dataset contains explicit categories such as `no parking` and `no laundry on site`.

Therefore, missing categorical values are treated as a separate `Unknown` category. This preserves the distinction between an unspecified facility and an explicitly unavailable facility.

In [9]:
# Categorical columns with missing values
categorical_missing_features = [
    "laundry_options",
    "parking_options"
]

# Replace missing categorical values with an explicit category
for column in categorical_missing_features:
    df[column] = df[column].fillna("Unknown")

# Verify that the categorical missing values have been handled
print("Remaining missing values in categorical features:")

for column in categorical_missing_features:
    print(f"{column}: {df[column].isna().sum()}")

Remaining missing values in categorical features:
laundry_options: 0
parking_options: 0


### 5.2 Investigate Missing Geographic Coordinates

The `lat` and `long` features contain a small number of missing values.

Because latitude and longitude represent a geographic coordinate pair, the missingness pattern is investigated before deciding how to handle these records.

The objective is to determine whether one coordinate is available when the other is missing.

In [10]:
# Check whether latitude and longitude are missing independently or as a pair
lat_missing = df["lat"].isna()
long_missing = df["long"].isna()

print("Both latitude and longitude missing:",
      (lat_missing & long_missing).sum())

print("Latitude missing but longitude available:",
      (lat_missing & ~long_missing).sum())

print("Longitude missing but latitude available:",
      (~lat_missing & long_missing).sum())

Both latitude and longitude missing: 1909
Latitude missing but longitude available: 0
Longitude missing but latitude available: 0


### 5.3 Verify Missing-Value Handling

The missing categorical values in `laundry_options` and `parking_options` were replaced with an `Unknown` category.

The missing latitude and longitude values were retained because they occur as complete coordinate pairs and represent unavailable geographic information rather than invalid coordinates.

A final missing-value check is performed before continuing to the next preprocessing stage.

In [11]:
# Check all remaining missing values after categorical handling
remaining_missing = df.isnull().sum()

# Display only columns that still contain missing values
remaining_missing = remaining_missing[remaining_missing > 0]

print("Remaining missing values:")
print(remaining_missing)

Remaining missing values:
lat     1909
long    1909
dtype: int64


## 6. Investigate Suspicious Bedroom and Bathroom Values

The EDA identified unusually large values for `beds` and `baths`.

Most listings contain realistic residential bedroom and bathroom counts, while a small number of records contain extremely large values such as 1,000 or 1,100 bedrooms and 25, 35, or 75 bathrooms.

These observations are investigated together with other property attributes to determine whether they represent plausible properties or likely data-entry/listing errors.

Extreme observations are not removed solely because they are statistically unusual.

In [12]:
# Inspect records with suspiciously large bedroom or bathroom counts
suspicious_beds_baths = df[
    (df["beds"] > 8) |
    (df["baths"] > 8.5)
]

print("Number of suspicious records:", len(suspicious_beds_baths))

print("\nSuspicious records:")
print(
    suspicious_beds_baths[
        ["price", "type", "sqfeet", "beds", "baths", "region", "state"]
    ].sort_values(
        ["beds", "baths"],
        ascending=False
    ).to_string(index=True)
)

Number of suspicious records: 4

Suspicious records:
        price       type  sqfeet  beds  baths        region state
89776    2449  apartment    1000  1100   75.0       chicago    il
90779    2449  apartment    1000  1100   75.0       chicago    il
37695    1500       land     100  1000   35.0  florida keys    fl
235159    550  apartment     250  1000   25.0    youngstown    oh


### 6.2 Remove Clearly Implausible Property Records

Four records contain extremely large bedroom and bathroom counts that are inconsistent with their reported property size and rental price.

For example, some records report 1,000 or 1,100 bedrooms for properties with only 100–1,000 square feet.

These observations are treated as likely data-entry or listing-data errors rather than legitimate extreme properties. Therefore, the four identified records are removed from the dataset.

In [13]:
# Identify clearly implausible bedroom/bathroom records
invalid_beds_baths = (
    (df["beds"] > 8) |
    (df["baths"] > 8.5)
)

print("Records to remove:", invalid_beds_baths.sum())

# Remove the identified records
df = df.loc[~invalid_beds_baths].copy()

print("Dataset shape after removing suspicious records:", df.shape)

# Verify that no suspicious bedroom/bathroom records remain
print(
    "Remaining records with beds > 8 or baths > 8.5:",
    ((df["beds"] > 8) | (df["baths"] > 8.5)).sum()
)

Records to remove: 4
Dataset shape after removing suspicious records: (383614, 17)
Remaining records with beds > 8 or baths > 8.5: 0


## 7. Investigate Extreme Square-Footage Values

The EDA showed that square footage is strongly right-skewed.

Most listings have relatively small residential property sizes, while a small number of records contain extremely large square-footage values.

These observations are investigated before deciding whether they should be removed, transformed, or retained. Statistical extremeness alone is not considered sufficient evidence that a record is invalid.

In [14]:
# Inspect the most extreme square-footage records
extreme_sqfeet = (
    df[["price", "type", "sqfeet", "beds", "baths", "region", "state"]]
    .sort_values("sqfeet", ascending=False)
    .head(20)
)

print(extreme_sqfeet.to_string(index=True))

        price       type   sqfeet  beds  baths                   region state
159607    750  apartment  8388607     2    1.0                  mankato    mn
356695   1025  apartment  8000000     2    1.0       anchorage / mat-su    ak
6608     7250      house  1019856     4    2.0            santa barbara    ca
6607    11350      house  1019856     5    4.0            santa barbara    ca
215904    625     duplex   999999     1    1.0                 syracuse    ny
26878     750  apartment   999999     1    1.0               eastern CT    ct
140646   2350  apartment   999999     1    1.0                   boston    ma
215901    500  apartment   999999     1    1.0                 syracuse    ny
63617    1090  apartment   775814     1    1.0           tampa bay area    fl
85613     725  apartment    99204     1    1.0  spokane / coeur d'alene    id
86381    1295      condo    99202     1    1.0  spokane / coeur d'alene    id
11868    1785  apartment    95242     2    2.0                 s

### 7.1 Analyze Extreme Square-Footage Values Using Percentiles

The `sqfeet` variable is strongly right-skewed, with a small number of extremely large values.

Instead of selecting an arbitrary fixed threshold, percentile-based statistics are used to understand the upper tail of the distribution.

This helps determine whether the extreme values are isolated observations or represent a substantial portion of the dataset.

In [15]:
# Calculate high percentiles of square footage
sqfeet_percentiles = df["sqfeet"].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999, 0.9999]
)

print("Square-footage percentiles:")
print(sqfeet_percentiles)

Square-footage percentiles:
0.9000     1400.000
0.9500     1644.000
0.9900     2403.000
0.9950     2800.000
0.9990     4200.000
0.9999    18229.932
Name: sqfeet, dtype: float64


### 7.2 Investigate the Extreme Upper Tail

The percentile analysis shows that square footage increases sharply in the extreme upper tail.

The 99.9th percentile is approximately 4,200 square feet, while the 99.99th percentile is approximately 18,230 square feet.

The observations above the 99.9th percentile are therefore examined to determine whether they represent legitimate large properties or suspicious/placeholder values.

In [16]:
# Calculate the data-driven 99.9th percentile
sqfeet_99_9 = df["sqfeet"].quantile(0.999)

# Select observations in the extreme upper tail
extreme_sqfeet = df[df["sqfeet"] > sqfeet_99_9].copy()

print("99.9th percentile:", sqfeet_99_9)
print("Records above 99.9th percentile:", len(extreme_sqfeet))

print("\nExtreme square-footage values:")
print(
    extreme_sqfeet[
        ["price", "type", "sqfeet", "beds", "baths", "region", "state"]
    ]
    .sort_values("sqfeet", ascending=False)
    .head(30)
    .to_string(index=True)
)

99.9th percentile: 4200.0
Records above 99.9th percentile: 380

Extreme square-footage values:
        price          type   sqfeet  beds  baths                    region state
159607    750     apartment  8388607     2    1.0                   mankato    mn
356695   1025     apartment  8000000     2    1.0        anchorage / mat-su    ak
6608     7250         house  1019856     4    2.0             santa barbara    ca
6607    11350         house  1019856     5    4.0             santa barbara    ca
140646   2350     apartment   999999     1    1.0                    boston    ma
26878     750     apartment   999999     1    1.0                eastern CT    ct
215904    625        duplex   999999     1    1.0                  syracuse    ny
215901    500     apartment   999999     1    1.0                  syracuse    ny
63617    1090     apartment   775814     1    1.0            tampa bay area    fl
85613     725     apartment    99204     1    1.0   spokane / coeur d'alene    id
863

### 7.3 Investigate Repeated Extreme Square-Footage Values

The extreme upper tail contains several unusually large and repeated square-footage values.

Some values, such as `999999`, appear to resemble placeholder or default entries rather than realistic residential floor areas.

The frequency of repeated extreme values is therefore examined before deciding whether any observations should be treated as invalid.

In [17]:
# Count repeated square-footage values in the extreme upper tail
extreme_sqfeet_counts = (
    extreme_sqfeet["sqfeet"]
    .value_counts()
    .head(20)
)

print("Most frequent square-footage values in the extreme tail:")
print(extreme_sqfeet_counts)

Most frequent square-footage values in the extreme tail:
sqfeet
5500      20
5000      19
7000      18
6000      12
5400      11
4385       6
4500       6
4300       6
11000      6
7500       5
4400       5
5300       5
4350       5
5200       5
50000      5
10000      4
999999     4
4600       4
5550       4
5800       4
Name: count, dtype: int64


### 7.4 Investigate Potential Placeholder Square-Footage Values

The extreme-tail analysis showed that many large square-footage values occur repeatedly and may represent legitimate large properties.

However, some values have patterns that suggest possible placeholder or data-entry artifacts, such as `999999`, `8000000`, and `8388607`.

These specific values are investigated separately rather than removing all extreme observations using an arbitrary threshold.

In [18]:
# Potentially suspicious square-footage values identified during EDA
suspicious_sqfeet_values = [999999, 8000000, 8388607]

for value in suspicious_sqfeet_values:
    matching_rows = df[df["sqfeet"] == value]

    print(f"\nSquare footage = {value:,}")
    print("Number of records:", len(matching_rows))

    if len(matching_rows) > 0:
        print(
            matching_rows[
                ["price", "type", "sqfeet", "beds", "baths", "region", "state"]
            ].to_string(index=True)
        )


Square footage = 999,999
Number of records: 4
        price       type  sqfeet  beds  baths      region state
26878     750  apartment  999999     1    1.0  eastern CT    ct
140646   2350  apartment  999999     1    1.0      boston    ma
215901    500  apartment  999999     1    1.0    syracuse    ny
215904    625     duplex  999999     1    1.0    syracuse    ny

Square footage = 8,000,000
Number of records: 1
        price       type   sqfeet  beds  baths              region state
356695   1025  apartment  8000000     2    1.0  anchorage / mat-su    ak

Square footage = 8,388,607
Number of records: 1
        price       type   sqfeet  beds  baths   region state
159607    750  apartment  8388607     2    1.0  mankato    mn


In [20]:
invalid_sqfeet_values = [
    999999,
    8000000,
    8388607
]

invalid_sqfeet = df["sqfeet"].isin(invalid_sqfeet_values)

print("Records to remove:", invalid_sqfeet.sum())

df = df.loc[~invalid_sqfeet].copy()

print("Dataset shape after removing square-footage artifacts:", df.shape)

print(
    "\nRemaining records with identified invalid square-footage values:",
    df["sqfeet"].isin(invalid_sqfeet_values).sum()
)

Records to remove: 6
Dataset shape after removing square-footage artifacts: (383608, 17)

Remaining records with identified invalid square-footage values: 0


## 8. Investigate Extreme Target Values

The target variable `price` is strongly right-skewed.

Most rental prices are concentrated around the lower range, while a small number of observations contain extremely large values.

Because `price` is the prediction target, extreme values are not automatically removed. They are first investigated to determine whether they represent legitimate high-priced listings or potential data-quality issues.

### 8.1 Investigate Extremely High Target Values

The EDA identified 14 listings with rental prices above $1 million.

Rather than removing high-priced listings using an arbitrary statistical threshold, these extreme observations are inspected individually to distinguish potential data-entry errors from legitimate high-value properties.

In [19]:
# Remove extreme rental-price records identified during EDA

extreme_price = df["price"] > 1_000_000

print("Records to remove:", extreme_price.sum())

df = df.loc[~extreme_price].copy()

print("Dataset shape after price cleaning:", df.shape)
print("Maximum remaining price:", df["price"].max())

Records to remove: 14
Dataset shape after price cleaning: (383600, 17)
Maximum remaining price: 885000


In [20]:
price_tail = df[df["price"] > 4200]

print("Records above 99.9th percentile:", len(price_tail))

print("\nPrice summary of the extreme tail:")
print(price_tail["price"].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

Records above 99.9th percentile: 1707

Price summary of the extreme tail:
count      1707.000000
mean      31210.275923
std      114788.794067
min        4202.000000
25%        4510.000000
50%        5200.000000
75%        6866.000000
90%       16485.600000
95%       79785.300000
99%      679710.000000
max      885000.000000
Name: price, dtype: float64


## 8. Categorical Encoding

The dataset contains several categorical features:

- `type`
- `region`
- `state`
- `laundry_options`
- `parking_options`

These features contain text categories and therefore cannot be directly used by most machine learning algorithms.

Before selecting an encoding technique, we examine the number of unique categories in each feature.

This is important because the number of categories affects the choice of encoding method. Features with a small number of categories can be safely one-hot encoded, while high-cardinality features such as `region` may require a more suitable encoding approach.

The encoding method will be selected based on the observed cardinality of each feature and applied later within the preprocessing pipeline to avoid data leakage.

In [21]:
# Check the number of unique categories in each categorical feature

for column in categorical_features:
    print(f"{column}: {df[column].nunique()} unique categories")

type: 12 unique categories
region: 404 unique categories
state: 51 unique categories
laundry_options: 6 unique categories
parking_options: 8 unique categories


## 8.1 Encoding Strategy

Based on the observed cardinality of the categorical features, different encoding approaches are selected.

- `type`, `laundry_options`, and `parking_options` have relatively few categories, so one-hot encoding is appropriate.
- `state` contains 51 categories and can still be handled using one-hot encoding.
- `region` contains 404 categories. One-hot encoding would create a large number of additional features, so target encoding is considered more suitable.

Target encoding must be performed in a leakage-safe manner because it uses the target variable (`price`). The encoding statistics for `region` will therefore be learned only from the training data and then applied to the validation and test data.

## 9. Numerical Feature Preprocessing

The numerical features in the dataset are:

- `sqfeet`
- `beds`
- `baths`
- `lat`
- `long`

EDA showed that `lat` and `long` contain missing values, while the other numerical features do not have missing values after the data-cleaning steps.

The missing latitude and longitude values will not be removed because only a small number of records are affected. Instead, missing values will be imputed during preprocessing.

The imputation values will be learned from the training data only to prevent data leakage.

The numerical features may also have different scales and distributions. Scaling will therefore be handled within the preprocessing pipeline for algorithms that require it.

In [22]:
# Check missing values in the numerical features

print("Missing values in numerical features:\n")

for column in numerical_features:
    print(f"{column}: {df[column].isna().sum()}")

Missing values in numerical features:

sqfeet: 0
beds: 0
baths: 0
lat: 1909
long: 1909


## 9.1 Numerical Imputation and Scaling Decision

The EDA identified 1,909 records with missing latitude and longitude values. In every case, both coordinates were missing together.

These records will be retained because the number of affected observations is small and removing them would unnecessarily reduce the dataset.

Missing numerical values will be imputed using the median calculated from the training data. Median imputation is selected because the numerical variables, particularly geographic and size-related variables, may contain skewed distributions and extreme values.

Numerical features will be standardized when required by the machine learning algorithm. The scaling parameters will be learned only from the training data to prevent data leakage.

## 10. Feature Engineering

Based on the EDA, the dataset already contains meaningful property characteristics such as square footage, bedrooms, bathrooms, amenities, and geographic information.

Although ratio-based features such as square-feet-per-bedroom could be created, these may introduce issues for properties with zero bedrooms, such as studio apartments.

Therefore, no additional ratio-based features are introduced at this stage. The original features will be retained, and feature usefulness will be evaluated during feature selection and model experimentation.

This approach avoids unnecessary feature creation while keeping the preprocessing process interpretable and defensible.

## 11. Feature Selection

Feature selection was guided by the EDA findings and the potential predictive value of each variable.

The `id`, URL-related columns, and `description` field were excluded because they were identifiers, metadata, or potential sources of target leakage.

EDA showed strong relationships among `sqfeet`, `beds`, and `baths`, indicating some redundancy between these variables. However, these features represent different property characteristics and may provide useful information to different machine learning algorithms.

Therefore, `sqfeet`, `beds`, and `baths` are retained for the initial modeling stage rather than being removed solely based on correlation.

Feature selection will be further evaluated during model development using model performance and feature-importance or selection techniques.

## 12. Repeated Listing Pattern Handling

EDA identified substantial repetition in the dataset. Property patterns were examined using the combination of `price`, `type`, `sqfeet`, `beds`, `baths`, `state`, `lat`, and `long`.

The analysis identified 43,364 repeated property-pattern groups containing 247,061 records.

Because identical property-pattern records can give repeated observations disproportionate influence during model training and can cause overly optimistic evaluation if distributed across different data splits, exact repeated patterns are reduced to one representative record per group.

This is applied only to exact repeated combinations of the selected variables. Records that are merely similar but not identical are retained.

After this step, the dataset contains 179,897 records representing unique property-price patterns.

In [24]:
# Define the exact property-price pattern used for deduplication.
# Keeping this definition in this notebook makes it reproducible in a fresh kernel.
near_duplicate_columns = [
    "price", "type", "sqfeet", "beds", "baths",
    "state", "lat", "long"
]

In [25]:
# Keep one representative record from each exact property-pattern group

before_deduplication = len(df)

df = (
    df.drop_duplicates(subset=near_duplicate_columns, keep="first")
      .copy()
)

after_deduplication = len(df)

print("Records before deduplication:", before_deduplication)
print("Records removed:", before_deduplication - after_deduplication)
print("Records after deduplication:", after_deduplication)

Records before deduplication: 383600
Records removed: 203697
Records after deduplication: 179903


## 13. Final Data Quality Check

After the cleaning, missing-value handling, outlier treatment, and repeated-pattern reduction, a final data-quality check is performed.

The purpose is to confirm that:

- No invalid target values remain.
- No invalid square-footage values remain.
- No invalid latitude values remain.
- The previously identified extreme `beds` and `baths` values have been removed.
- Expected missing values remain only in `lat` and `long`.

In [26]:
# Final data-quality checks

print("Dataset shape:", df.shape)

print("\nInvalid price records:",
      (df["price"] <= 0).sum())

print("Invalid square-footage records:",
      (df["sqfeet"] <= 0).sum())

invalid_lat = (
    df["lat"].notna() &
    ~df["lat"].between(-90, 90)
)

print("Invalid latitude records:",
      invalid_lat.sum())

print("Suspicious beds/baths records:",
      ((df["beds"] > 8) | (df["baths"] > 8.5)).sum())

print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

Dataset shape: (179903, 17)

Invalid price records: 0
Invalid square-footage records: 0
Invalid latitude records: 0
Suspicious beds/baths records: 0

Missing values:
lat     1094
long    1094
dtype: int64


## 13. Train-Test Split

After cleaning and reducing exact repeated property-pattern records, the dataset is divided into training and test sets.

An 80/20 split is used:

- 80% for training and model development
- 20% reserved for final evaluation

The training set will be used for cross-validation, preprocessing, model training, and hyperparameter tuning.

The test set will remain unseen until the final evaluation of the selected model.

All preprocessing steps that learn parameters from the data, such as imputation, scaling, and target encoding, will be fitted using the training data only.

In [27]:
from sklearn.model_selection import train_test_split

# Separate predictors and target
X = df.drop(columns=["price"])
y = df["price"]

# Split into 80% training and 20% test data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (143922, 16) (143922,)
Test set: (35981, 16) (35981,)


In [28]:
# ============================================================
# SAVE RAW TRAIN / TEST SPLITS FOR MODEL DEVELOPMENT
# ============================================================

# Create copies of the original training and testing features
# before target encoding and other learned transformations.
raw_train_df = X_train.copy()
raw_test_df = X_test.copy()

# Add the original target variable back to each dataset
raw_train_df["price"] = y_train.values
raw_test_df["price"] = y_test.values

# Save the raw splits for use in model development
raw_train_df.to_csv("../data/raw_train.csv", index=False)
raw_test_df.to_csv("../data/raw_test.csv", index=False)

print("Raw train/test splits saved successfully.")
print("Raw training data:", raw_train_df.shape)
print("Raw testing data :", raw_test_df.shape)

Raw train/test splits saved successfully.
Raw training data: (143922, 17)
Raw testing data : (35981, 17)


## 14. Preprocessing Pipeline Preparation

The preprocessing pipeline is designed to apply the decisions made during EDA and data cleaning.

Numerical features will use median imputation for missing values and scaling where required.

Binary features are already represented using 0 and 1 and therefore do not require encoding.

Categorical features with relatively low cardinality will be one-hot encoded.

The high-cardinality `region` feature will be handled separately using a leakage-safe target encoding approach. Target encoding will use training data only and will be performed carefully during cross-validation to avoid target leakage.

In [31]:
# Define feature groups for the preprocessing pipeline

numerical_features = [
    "sqfeet",
    "beds",
    "baths",
    "lat",
    "long"
]

binary_features = [
    "cats_allowed",
    "dogs_allowed",
    "smoking_allowed",
    "wheelchair_access",
    "electric_vehicle_charge",
    "comes_furnished"
]

onehot_features = [
    "type",
    "state",
    "laundry_options",
    "parking_options"
]

target_encode_features = [
    "region"
]

print("Numerical features:", numerical_features)
print("Binary features:", binary_features)
print("One-hot encoded features:", onehot_features)
print("Target encoded features:", target_encode_features)

Numerical features: ['sqfeet', 'beds', 'baths', 'lat', 'long']
Binary features: ['cats_allowed', 'dogs_allowed', 'smoking_allowed', 'wheelchair_access', 'electric_vehicle_charge', 'comes_furnished']
One-hot encoded features: ['type', 'state', 'laundry_options', 'parking_options']
Target encoded features: ['region']


## 14.1 Numerical and Categorical Preprocessing

The preprocessing transformations are applied separately according to feature type.

For numerical features, missing values will be replaced using the median calculated from the training data, followed by standardization.

Binary features are already encoded as 0 and 1, so they can be passed through without additional encoding.

Low-cardinality categorical features will be one-hot encoded. Unknown categories that may appear in unseen data will be handled safely by the encoder.

The `region` feature will be processed separately using leakage-safe target encoding because its encoding depends on the target variable.

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Add the leakage-safe region encoding to the numerical features.
# region_encoded is already a numerical value.
processed_numerical_features = numerical_features + ["region_encoded"]

# Numerical preprocessing:
# 1. Fill missing values using the median
# 2. Standardize the numerical features
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing:
# 1. Fill any unexpected missing values
# 2. Convert categories into one-hot encoded columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine all preprocessing transformations
preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        processed_numerical_features
    ),
    (
        "categorical",
        categorical_pipeline,
        onehot_features
    ),
    (
        "binary",
        "passthrough",
        binary_features
    )
])

print("Preprocessing pipeline created successfully.")
print("Numerical features:", processed_numerical_features)

Preprocessing pipeline created successfully.
Numerical features: ['sqfeet', 'beds', 'baths', 'lat', 'long', 'region_encoded']


## 14.2 Leakage-Safe Target Encoding for `region`

The `region` feature is target encoded because its categories can be represented using their relationship with rental price.

Since target encoding uses the target variable, directly calculating the encoding from all training observations would allow each training observation's target to influence its own encoded value.

To avoid this, **out-of-fold target encoding** is used for the training data. Each training observation is encoded using regional statistics calculated from other training folds rather than from the observation itself.

After the out-of-fold encodings are created, a final region-to-price mapping is learned from the complete training data. This final mapping is used to transform the unseen test data.

Smoothing is applied to reduce the influence of regions with very few observations.

The test target values are never used when creating the encoding.

In [33]:
from sklearn.model_selection import KFold

# Number of folds used for out-of-fold encoding
n_splits = 5

# Smoothing reduces the effect of regions with very few observations
smoothing = 10

# Create 5 folds from the training data only
kf = KFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

# Create an empty Series to store the encoded value
# for every training row
region_encoded_train = pd.Series(
    index=X_train.index,
    dtype=float
)

# ---------------------------------------------------------
# OUT-OF-FOLD TARGET ENCODING
# ---------------------------------------------------------
# Each training row is encoded using information from
# the other folds, so its own target value is NOT used.
# This prevents target leakage.
# ---------------------------------------------------------

for train_idx, validation_idx in kf.split(X_train):

    # Training portion of the current fold
    fold_X_train = X_train.iloc[train_idx]
    fold_y_train = y_train.iloc[train_idx]

    # Validation portion of the current fold
    fold_X_validation = X_train.iloc[validation_idx]

    # Mean target value calculated only from the
    # current fold's training data
    fold_global_mean = fold_y_train.mean()

    # Calculate mean price and number of records
    # for each region using only the fold's training data
    fold_region_stats = (
        pd.DataFrame({
            "region": fold_X_train["region"].values,
            "price": fold_y_train.values
        })
        .groupby("region")["price"]
        .agg(["mean", "count"])
    )

    # Apply smoothing to make small groups more reliable
    fold_region_stats["encoded_value"] = (
        (
            fold_region_stats["count"] * fold_region_stats["mean"]
            + smoothing * fold_global_mean
        )
        / (fold_region_stats["count"] + smoothing)
    )

    # Encode the validation portion of this fold
    # using ONLY the other four folds
    encoded_values = (
        fold_X_validation["region"]
        .map(fold_region_stats["encoded_value"])
        .fillna(fold_global_mean)
    )

    # Store the encoded values in their original positions
    region_encoded_train.iloc[validation_idx] = encoded_values.values


# ---------------------------------------------------------
# FINAL REGION MAPPING FOR THE TEST SET
# ---------------------------------------------------------
# Now use ALL training data to create the final mapping.
# This mapping will be applied to the test set only.
# y_test is never used here.
# ---------------------------------------------------------

global_mean = y_train.mean()

region_stats = (
    pd.DataFrame({
        "region": X_train["region"].values,
        "price": y_train.values
    })
    .groupby("region")["price"]
    .agg(["mean", "count"])
)

# Apply smoothing to the final training mapping
region_stats["encoded_value"] = (
    (
        region_stats["count"] * region_stats["mean"]
        + smoothing * global_mean
    )
    / (region_stats["count"] + smoothing)
)

# Create dictionary for encoding the test set
region_encoding = region_stats["encoded_value"].to_dict()

# Encode test regions using the training-data mapping
region_encoded_test = (
    X_test["region"]
    .map(region_encoding)
    .fillna(global_mean)
)

print("Leakage-safe region target encoding created successfully.")
print("Training rows encoded:", len(region_encoded_train))
print("Unique regions in training:", len(region_encoding))
print("Global training mean price:", round(global_mean, 2))
print("Missing encoded training values:", region_encoded_train.isna().sum())
print("Missing encoded test values:", region_encoded_test.isna().sum())

Leakage-safe region target encoding created successfully.
Training rows encoded: 143922
Unique regions in training: 404
Global training mean price: 1422.04
Missing encoded training values: 0
Missing encoded test values: 0


## 14.3 Applying the Region Target Encoding

The target encoding learned from the training data is now applied to both the training and test datasets.

The same mapping learned from the training data is used for both datasets. The test target values are not used during this process.

If a region appears in the test set but was not present in the training set, the global training-set mean is used as the fallback value.

The original `region` column is then removed because its information has been converted into a numerical encoded feature.

In [34]:
# Create copies so the original X_train and X_test remain unchanged
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# Add the leakage-safe region encoding
# Training values come from out-of-fold encoding
X_train_encoded["region_encoded"] = region_encoded_train

# Test values use the final mapping learned from ALL training data
X_test_encoded["region_encoded"] = region_encoded_test

# Remove the original categorical region column
# because its information has now been converted
# into the numerical region_encoded feature
X_train_encoded = X_train_encoded.drop(columns=["region"])
X_test_encoded = X_test_encoded.drop(columns=["region"])

print("Region encoding applied successfully.")
print("Training data shape:", X_train_encoded.shape)
print("Test data shape:", X_test_encoded.shape)
print("Missing encoded training values:",
      X_train_encoded["region_encoded"].isna().sum())
print("Missing encoded test values:",
      X_test_encoded["region_encoded"].isna().sum())

Region encoding applied successfully.
Training data shape: (143922, 16)
Test data shape: (35981, 16)
Missing encoded training values: 0
Missing encoded test values: 0


## 14.4 Applying the Main Preprocessing Pipeline

The main preprocessing pipeline is now fitted using the training data only.

The pipeline performs:
- Median imputation for missing numerical values.
- Standardization of numerical features.
- One-hot encoding of categorical features.
- Passing binary features through unchanged.

The fitted pipeline is then used to transform both the training and test datasets.

The test dataset is only transformed using the preprocessing rules learned from the training dataset. This prevents information from the test set from influencing the preprocessing process and helps prevent data leakage.

In [35]:
# Fit the preprocessing pipeline using TRAINING data only.
# This is important because the test set must remain unseen
# when learning preprocessing parameters.

preprocessor.fit(X_train_encoded)

# Transform the training data using the rules learned above
X_train_processed = preprocessor.transform(X_train_encoded)

# Transform the test data using the SAME rules learned
# from the training data
X_test_processed = preprocessor.transform(X_test_encoded)

print("Main preprocessing completed successfully.")
print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Main preprocessing completed successfully.
Processed training shape: (143922, 89)
Processed test shape: (35981, 89)


## 14.5 Preprocessing Verification

After preprocessing, the transformed training and test datasets are checked to confirm that:

- Both datasets contain the same number of features.
- No missing values remain in the processed data.
- The preprocessing has successfully converted the original categorical features into numerical representations.

This verification helps ensure that the datasets are ready for machine learning algorithms.

In [36]:
# Check the final processed dataset dimensions
print("Processed training data shape:", X_train_processed.shape)
print("Processed test data shape:", X_test_processed.shape)

# Check whether any missing values remain
print(
    "\nMissing values in training data:",
    np.isnan(X_train_processed).sum()
)

print(
    "Missing values in test data:",
    np.isnan(X_test_processed).sum()
)

Processed training data shape: (143922, 89)
Processed test data shape: (35981, 89)

Missing values in training data: 0
Missing values in test data: 0


## 14.6 Target Variable Transformation

The target variable `price` showed a strong right-skewed distribution during EDA, with a small number of very large rental prices.

Extreme skewness can make regression models focus heavily on high-value observations.

Therefore, the distribution of the target variable will be examined again after data cleaning. A logarithmic transformation will be considered if it helps reduce the skewness of the target distribution.

The transformation will be applied to the training target and consistently applied to the test target when evaluating models. The original price scale will be recovered using the inverse transformation when predictions need to be interpreted in dollars.

In [37]:
from scipy.stats import skew

# Calculate the skewness of the training target
# Skewness measures how asymmetric the distribution is.
# A large positive value means the distribution has a long
# right tail caused by very high price values.

price_skewness = skew(y_train)

print("Training target skewness:", round(price_skewness, 4))
print("Training target minimum:", y_train.min())
print("Training target maximum:", y_train.max())

Training target skewness: 83.833
Training target minimum: 1
Training target maximum: 859864


## 14.7 Evaluating a Logarithmic Target Transformation

The cleaned target variable is still highly right-skewed.

A logarithmic transformation is evaluated to determine whether it can reduce the effect of the long right tail and produce a more balanced target distribution.

The `log1p` transformation is used because the target contains positive values and this transformation safely handles values close to zero.

The transformation will only be adopted if it provides a meaningful reduction in skewness.

In [38]:
# Apply log1p transformation to the training target.
# log1p(x) calculates log(1 + x), which is safe for
# non-negative price values.

y_train_log = np.log1p(y_train)

# Calculate skewness after the transformation
log_price_skewness = skew(y_train_log)

print("Original target skewness:", round(price_skewness, 4))
print("Log-transformed target skewness:", round(log_price_skewness, 4))

Original target skewness: 83.833
Log-transformed target skewness: -1.7206


## 14.8 Final Target Transformation

The log transformation reduced the target skewness from 87.5835 to -1.6232.

Based on this substantial reduction in skewness, the logarithmic transformation will be used for model training.

The original target values will be retained for evaluation and interpretation.

During prediction, the inverse transformation (`expm1`) will be used to convert predictions back to the original rental-price scale.

In [39]:
# The log transformation substantially reduced the skewness
# of the rental price distribution.
#
# Therefore, the log-transformed target will be used
# during model training.
#
# The original target values are kept for final evaluation
# so that model performance can be reported in actual
# rental-price units.

y_train_log = np.log1p(y_train)

# Keep the original test prices for final evaluation
y_test_original = y_test.copy()

print("Final target transformation completed.")
print("Training target shape:", y_train_log.shape)
print("Test target shape:", y_test_original.shape)

Final target transformation completed.
Training target shape: (143922,)
Test target shape: (35981,)


## 14.9 Exporting the Processed Training and Test Datasets

The final processed training and test datasets are exported as CSV files for use during model development.

The training dataset contains the processed features and the log-transformed rental price target.

The test dataset contains the same processed features and the original rental price target so that model predictions can later be evaluated in the original dollar scale.

The same feature structure is maintained in both datasets.

In [40]:
# Convert the processed NumPy arrays back into DataFrames
# so that feature names are preserved.

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out()
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out()
)

# Add the target variables
# Training uses the log-transformed target
X_train_processed_df["price_log"] = y_train_log.values

# Test keeps the original price for evaluation
X_test_processed_df["price"] = y_test_original.values

# Save the processed datasets
X_train_processed_df.to_csv("../data/train.csv", index=False)
X_test_processed_df.to_csv("../data/test.csv", index=False)

print("Processed datasets saved successfully.")
print("Training dataset shape:", X_train_processed_df.shape)
print("Test dataset shape:", X_test_processed_df.shape)

Processed datasets saved successfully.
Training dataset shape: (143922, 90)
Test dataset shape: (35981, 90)
